# Optimizer-equipped CNN-MLP on CIC-DDoS2019

Replication and adaptation of Mehmood et al. (2025), PLOS ONE 20(1): e0312425.

This notebook is deliberately almost empty. It fetches a manifest of short-lived
presigned URLs, downloads the code bundle GitHub Actions built from the repository,
and hands control to `src/session.py`. Keeping the pipeline out of the notebook is
what stops the two from drifting apart.

**No AWS credentials are present here.** GitHub Actions is the only place that reads
the repository secrets. It signs the URLs; this notebook can only use them, and only
until they expire.

Settings this notebook requires: **Accelerator = None** (the run is CPU-only, matching
the paper's own non-CUDA hardware) and **Internet = on** (for the presigned URLs).
The dataset is read from `/kaggle/input`, not from S3.

In [ ]:
# This cell is REPLACED by GitHub Actions before the kernel is pushed.
# The committed version holds no URL, and the injected version is never committed.
PRESIGNED_MANIFEST_URL = ""
RUN_ID = ""
INPUT_ROOT = "/kaggle/input/cicddos2019-parquet"

In [ ]:
import subprocess, sys

# Kaggle's CPU image already has torch, numpy, pandas, pyarrow, sklearn and shap.
for package in ("optuna", "lightgbm"):
    try:
        __import__(package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import torch
print("torch", torch.__version__, "| cuda visible:", torch.cuda.is_available())
assert not torch.cuda.is_available(), "set Accelerator to None: this experiment is CPU-only"

In [ ]:
import io, json, sys, tarfile
from pathlib import Path

import requests

assert PRESIGNED_MANIFEST_URL, "no manifest URL: this notebook must be launched by the workflow"

CODE = Path("/kaggle/working/code")
CODE.mkdir(parents=True, exist_ok=True)

# Minimal bootstrap: fetch the manifest, pull the code bundle, unpack it.
# Everything after this line runs the repository's own code.
manifest = requests.get(PRESIGNED_MANIFEST_URL, timeout=60).json()
bundle_key = next(k for k in manifest["entries"] if k.endswith("code_bundle.tar.gz"))
bundle = requests.get(manifest["entries"][bundle_key]["get_url"], timeout=300).content
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    archive.extractall(CODE)

sys.path.insert(0, str(CODE / "src"))
print(f"run {manifest['run_id']} | phase {manifest.get('phase')} | "
      f"{len(manifest['entries'])} keys | expires {manifest['expires_at']}")
print("code:", sorted(p.name for p in (CODE / 'src').glob('*.py')))

In [ ]:
import session

exit_code = session.run_session(
    manifest_url=PRESIGNED_MANIFEST_URL,
    work_dir=Path("/kaggle/working/run"),
    repo_root=CODE,
    input_root=Path(INPUT_ROOT),
    run_id=RUN_ID,
)

# Exit 0 covers 'finished the phase' and 'ran out of session time with a valid
# checkpoint'. Kaggle cancelling on the 12 hour limit is an expected outcome,
# not a failure, so long as the checkpoint uploaded.
print("exit code:", exit_code)
assert exit_code == 0, f"session failed with {exit_code}"

In [ ]:
# Progress, for reading in the Kaggle output. No URLs are printed anywhere:
# a presigned query string carries X-Amz-Signature.
state_path = Path("/kaggle/working/run/checkpoints/training_state.json")
if state_path.exists():
    state = json.loads(state_path.read_text())
    print(f"phase   : {state['phase']}")
    print(f"epoch   : {state['current_epoch']} / {state['total_epochs']}")
    print(f"status  : {state['status']}")
    print(f"sessions: {len(state.get('sessions', []))}")
else:
    print("no training state yet: this session ran a preparation phase")